# Lab 19: API Requests and Caching — Analysis

Run these experiments to see the difference caching makes.

**Before you start:** Make sure your `crypto.py` implementations pass all tests.

**Setup:** Put your CoinGecko Demo API key in the cell below.

In [4]:
import sys
sys.path.insert(0, '../src')

import time
from crypto import get_price, get_prices_batch, CoinCache, get_price_cached

API_KEY = "CG-2YfSPdUEW8ihDchSj4qanBvf"  # Replace with your CoinGecko Demo key

## Experiment 1: Uncached vs. Cached

Fetch Bitcoin's price 10 times — first without caching, then with caching.
Compare the total time and number of API calls.

In [5]:
# --- Uncached: 10 direct API calls ---
start = time.time()
for i in range(10):
    price = get_price("bitcoin", API_KEY)
uncached_time = time.time() - start

print(f"Uncached: 10 requests in {uncached_time:.2f} seconds")
print(f"Last price: ${price:,.2f}")

Uncached: 10 requests in 1.45 seconds
Last price: $72,384.00


In [6]:
# --- Cached: 10 lookups through cache ---
cache = CoinCache(ttl_seconds=60)

start = time.time()
for i in range(10):
    price = get_price_cached("bitcoin", API_KEY, cache)
cached_time = time.time() - start

print(f"Cached: 10 lookups in {cached_time:.2f} seconds")
print(f"Cache hits: {cache.hits}, Cache misses: {cache.misses}")
print(f"Speedup: {uncached_time / cached_time:.1f}x faster")

Cached: 10 lookups in 0.06 seconds
Cache hits: 9, Cache misses: 1
Speedup: 23.3x faster


### Writeup Questions — Experiment 1

1. How many API calls did the cached version actually make? Why that number?
2. What was the approximate speedup? Why is the difference so large?
3. Is there any downside to this speedup? What are you giving up?


*Your answers here:*

1. It made 1 actual API calls, it made 1 beacuse after one it is stored in your cache for quicker access.
2. The speedup was 23.3x faster, it is so large because accessing data from your cache is much more efficent and quicker.
3. You are giving up recency of data as you yourself specify how often you want the data to be updated.

## Experiment 2: TTL Exploration

Try three different TTL values. For each one, simulate a pattern
of lookups spaced 2 seconds apart and observe the hit rate.

In [7]:
ttl_values = [1, 5, 30]

for ttl in ttl_values:
    cache = CoinCache(ttl_seconds=ttl)
    
    for i in range(6):
        price = get_price_cached("bitcoin", API_KEY, cache)
        if i < 5:  # Don't sleep after last lookup
            time.sleep(2)
    
    total = cache.hits + cache.misses
    hit_rate = cache.hits / total * 100
    print(f"TTL={ttl:2d}s: {cache.hits} hits, {cache.misses} misses, hit rate={hit_rate:.0f}%")

TTL= 1s: 0 hits, 6 misses, hit rate=0%
TTL= 5s: 4 hits, 2 misses, hit rate=67%
TTL=30s: 5 hits, 1 misses, hit rate=83%


### Writeup Questions — Experiment 2

1. With TTL=1 second and lookups every 2 seconds, what hit rate do you expect? Did the results match?
2. If you were building a portfolio tracker that updates every time you open the app, what TTL would you choose? Explain your reasoning.
3. Is there a scenario where you'd want a TTL of 0 (no caching at all)? What about a TTL of infinity (cache forever)?

*Your answers here:*

1. I expected 0 hits and a 0% hit rate, yes the results matched.
2. I would choose a ttl of 4 seconds, It allows you to check your portfolio every 4 seconds with new prices and execute precise trades with recent price data.
3. You would want to caching when using something like a life alert, with information that is crucially important to always have updated, also with a API that has no rate limits. You would want to cache forever for something with really expensive prices for exceeding your rate limit, or something like a birthday search tool and information that never changes.

## Experiment 3: Batch Efficiency

Compare fetching 5 coins one at a time vs. in a single batch request.

In [8]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "dogecoin"]

# --- Individual calls ---
start = time.time()
individual_prices = {}
for coin in coins:
    individual_prices[coin] = get_price(coin, API_KEY)
individual_time = time.time() - start

print(f"Individual: {len(coins)} calls in {individual_time:.2f} seconds")

# --- Batch call ---
start = time.time()
batch_prices = get_prices_batch(coins, API_KEY)
batch_time = time.time() - start

print(f"Batch: 1 call in {batch_time:.2f} seconds")
print(f"Speedup: {individual_time / batch_time:.1f}x faster")

# Show the prices
print("\nPrices:")
for coin, price in batch_prices.items():
    print(f"  {coin}: ${price:,.2f}")

Individual: 5 calls in 0.70 seconds
Batch: 1 call in 0.12 seconds
Speedup: 5.9x faster

Prices:
  bitcoin: $72,370.00
  ethereum: $2,214.51
  solana: $84.25
  cardano: $0.26
  dogecoin: $0.09


### Writeup Questions — Experiment 3

1. How much faster was the batch call? Where does that time saving come from?
2. Batching and caching are both ways to reduce API calls. When would you use one vs. the other? Can you use both?

*Your answers here:*

1. The batch call was 5 times faster, this time saving comes from not having to go back to your program 5 times to write the individual price and go back until through all of them. You grab all the data in one trip.
2. You use batching when you have lots of info at once you want to retreive. You use caching when you need to constantly go back and forth to the API looking for info, caching helps you not over exceed the rate limit. I would use both on a API that has a huge fee for exceeding rate limits, and when you have lots of info you need and it takes a long time to query the API.